# Housing Price Prediction — Advanced Regression ML Project

I built this notebook as a full end-to-end supervised machine learning project for predicting house prices.

The project focuses only on **regression**.  
No classification. No unsupervised learning. Just a clean, focused housing price prediction workflow.

## What this notebook includes

- Google Colab + Google Drive dataset loading
- Exploratory Data Analysis
- strong visualizations
- feature engineering
- preprocessing pipelines
- Linear Regression baseline
- Random Forest
- XGBoost
- feature importance
- feature selection
- model comparison
- residual analysis
- final business-style conclusions\n- cross-validation\n- XGBoost hyperparameter tuning\n- SHAP explainability\n- learning curves\n- saved production-ready model

Dataset path used in this project:

```python
/content/drive/MyDrive/Colab Notebooks/housing_iteration_6_regression.csv
```

## 1. Environment Setup

First, I mount Google Drive and load the libraries needed for the project.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

# Feature selection
from sklearn.feature_selection import SelectKBest, f_regression, SelectFromModel, RFE
from sklearn.feature_selection import VarianceThreshold

# XGBoost
try:
    from xgboost import XGBRegressor
except ImportError:
    import sys
    !{sys.executable} -m pip install xgboost -q
    from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

sns.set_theme(style="whitegrid")

## 2. Load the Dataset

I load the housing dataset directly from Google Drive.  
This keeps the notebook clean and reproducible in Google Colab.

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/housing_iteration_6_regression.csv"

housing = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print(f"Dataset shape: {housing.shape}")

housing.head()

## 3. Quick Dataset Overview

Before building any model, I want to understand what the dataset looks like.

In [ ]:
housing.info()

In [ ]:
housing.describe().T

In [ ]:
TARGET = "SalePrice"

if TARGET not in housing.columns:
    raise ValueError(f"Target column '{TARGET}' was not found in the dataset.")

print("Target column:", TARGET)
print("Missing values in target:", housing[TARGET].isna().sum())

## 4. Missing Values Overview

Missing values are not automatically bad, but I need to know where they are.  
This visualization helps me quickly identify the columns that need attention.

In [ ]:
missing = housing.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

plt.figure(figsize=(12, 7))
sns.barplot(x=missing.head(25).values, y=missing.head(25).index)
plt.title("Top Columns with Missing Values")
plt.xlabel("Number of Missing Values")
plt.ylabel("Column")
plt.tight_layout()
plt.show()

missing.head(25)

## 5. Feature Types

Here I check how many numeric and categorical variables the dataset contains.

In [ ]:
numeric_cols = housing.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = housing.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

feature_type_counts = pd.Series({
    "Numeric": len(numeric_cols),
    "Categorical": len(categorical_cols)
})

plt.figure(figsize=(6, 5))
sns.barplot(x=feature_type_counts.index, y=feature_type_counts.values)
plt.title("Numeric vs Categorical Columns")
plt.ylabel("Number of Columns")
plt.tight_layout()
plt.show()

feature_type_counts

## 6. Target Variable Analysis

`SalePrice` is the variable I want to predict.  
Understanding its distribution is super important before modeling.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(housing[TARGET], kde=True, bins=40)
plt.title("Distribution of SalePrice")
plt.xlabel("SalePrice")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(np.log1p(housing[TARGET]), kde=True, bins=40)
plt.title("Log-Transformed Distribution of SalePrice")
plt.xlabel("log1p(SalePrice)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 7. Correlation Heatmap

This gives me a first look at the strongest numeric relationships with house price.

In [ ]:
numeric_data = housing.select_dtypes(include=["int64", "float64"])

corr = numeric_data.corr(numeric_only=True)
top_corr_features = corr[TARGET].abs().sort_values(ascending=False).head(15).index

plt.figure(figsize=(12, 9))
sns.heatmap(
    housing[top_corr_features].corr(numeric_only=True),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)
plt.title("Correlation Heatmap — Top Features Related to SalePrice")
plt.tight_layout()
plt.show()

corr[TARGET].sort_values(ascending=False).head(15)

## 8. Strong Numeric Price Drivers

I visualize some of the most important numeric relationships with `SalePrice`.

In [ ]:
top_numeric_predictors = (
    corr[TARGET]
    .abs()
    .sort_values(ascending=False)
    .drop(TARGET)
    .head(6)
    .index
    .tolist()
)

for col in top_numeric_predictors:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=housing, x=col, y=TARGET, alpha=0.6)
    sns.regplot(data=housing, x=col, y=TARGET, scatter=False, color="red")
    plt.title(f"SalePrice vs {col}")
    plt.tight_layout()
    plt.show()

## 9. Categorical Feature Visualizations

Categorical variables can be very powerful in housing data.  
I focus on features with a reasonable number of categories so the charts stay readable.

In [ ]:
candidate_categorical = [
    col for col in categorical_cols
    if housing[col].nunique(dropna=True) <= 20
]

for col in candidate_categorical[:6]:
    order = housing.groupby(col)[TARGET].median().sort_values(ascending=False).index

    plt.figure(figsize=(12, 5))
    sns.boxplot(data=housing, x=col, y=TARGET, order=order)
    plt.title(f"SalePrice by {col}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 10. Neighborhood Price Analysis

Neighborhood is usually one of the most important housing market signals.  
This chart makes the price differences between neighborhoods very visible.

In [ ]:
if "Neighborhood" in housing.columns:
    neighborhood_prices = (
        housing.groupby("Neighborhood")[TARGET]
        .median()
        .sort_values(ascending=False)
    )

    plt.figure(figsize=(14, 7))
    sns.barplot(x=neighborhood_prices.values, y=neighborhood_prices.index)
    plt.title("Median SalePrice by Neighborhood")
    plt.xlabel("Median SalePrice")
    plt.ylabel("Neighborhood")
    plt.tight_layout()
    plt.show()

    neighborhood_prices.head(15)
else:
    print("Neighborhood column was not found in the dataset.")

## 11. Feature Engineering

I add a few simple engineered features that make sense for housing data.

These features try to capture:

- total usable space
- house age
- remodeling age
- total bathrooms
- porch/deck area
- whether the house has a garage

In [ ]:
df = housing.copy()

if {"TotalBsmtSF", "GrLivArea"}.issubset(df.columns):
    df["TotalLivingArea"] = df["TotalBsmtSF"] + df["GrLivArea"]

if {"YrSold", "YearBuilt"}.issubset(df.columns):
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

if {"YrSold", "YearRemodAdd"}.issubset(df.columns):
    df["YearsSinceRemodel"] = df["YrSold"] - df["YearRemodAdd"]

bath_cols = ["FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"]
if set(bath_cols).issubset(df.columns):
    df["TotalBathrooms"] = (
        df["FullBath"] 
        + 0.5 * df["HalfBath"]
        + df["BsmtFullBath"]
        + 0.5 * df["BsmtHalfBath"]
    )

porch_cols = ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "WoodDeckSF"]
existing_porch_cols = [col for col in porch_cols if col in df.columns]
if existing_porch_cols:
    df["TotalOutdoorArea"] = df[existing_porch_cols].sum(axis=1)

if "GarageArea" in df.columns:
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)

print("New dataset shape after feature engineering:", df.shape)

new_features = [
    col for col in [
        "TotalLivingArea", "HouseAge", "YearsSinceRemodel",
        "TotalBathrooms", "TotalOutdoorArea", "HasGarage"
    ]
    if col in df.columns
]

df[new_features + [TARGET]].head()

## 12. Engineered Feature Visualizations

Now I check whether the engineered features are actually meaningful.

In [ ]:
for col in new_features:
    if df[col].nunique() > 2:
        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=df, x=col, y=TARGET, alpha=0.6)
        sns.regplot(data=df, x=col, y=TARGET, scatter=False, color="red")
        plt.title(f"SalePrice vs Engineered Feature: {col}")
        plt.tight_layout()
        plt.show()
    else:
        plt.figure(figsize=(7, 5))
        sns.boxplot(data=df, x=col, y=TARGET)
        plt.title(f"SalePrice by {col}")
        plt.tight_layout()
        plt.show()

## 13. Train-Test Split

I separate features from the target and create a train-test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

## 14. Preprocessing Pipeline

The pipeline handles numeric and categorical features separately.

- numeric columns: median imputation + scaling
- categorical columns: most frequent imputation + one-hot encoding

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

## 15. Evaluation Function

I use MAE, RMSE and R² to evaluate models.

- **MAE**: average absolute error
- **RMSE**: penalizes larger mistakes more strongly
- **R²**: explains how much variance the model captures

In [ ]:
def evaluate_model(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Predictions": predictions,
        "Pipeline": model
    }

## 16. Model Training

I train three models:

1. Linear Regression as a simple baseline
2. Random Forest as a strong tree-based model
3. XGBoost as the advanced boosting model

In [ ]:
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

ridge_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10))
])

random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

xgboost_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

results = []

for name, model in [
    ("Linear Regression", linear_model),
    ("Ridge Regression", ridge_model),
    ("Random Forest", random_forest_model),
    ("XGBoost", xgboost_model)
]:
    print(f"Training {name}...")
    results.append(evaluate_model(name, model, X_train, X_test, y_train, y_test))

model_results = pd.DataFrame([
    {k: v for k, v in result.items() if k not in ["Predictions", "Pipeline"]}
    for result in results
]).sort_values("RMSE")

model_results

## 17. Model Comparison Visualizations

Now I compare the models visually.  
The lower the MAE/RMSE, the better.  
The higher the R², the better.

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=model_results, x="RMSE", y="Model")
plt.title("Model Comparison by RMSE")
plt.xlabel("RMSE")
plt.ylabel("")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=model_results.sort_values("MAE"), x="MAE", y="Model")
plt.title("Model Comparison by MAE")
plt.xlabel("MAE")
plt.ylabel("")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=model_results.sort_values("R2", ascending=False), x="R2", y="Model")
plt.title("Model Comparison by R²")
plt.xlabel("R²")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 18. Actual vs Predicted Prices

This plot shows how close the predictions are to the true sale prices.  
A perfect model would place every point exactly on the diagonal line.

In [ ]:
best_result = sorted(results, key=lambda x: x["RMSE"])[0]
best_model_name = best_result["Model"]
best_predictions = best_result["Predictions"]

plt.figure(figsize=(8, 8))
sns.scatterplot(x=y_test, y=best_predictions, alpha=0.65)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.title(f"Actual vs Predicted SalePrice — {best_model_name}")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.tight_layout()
plt.show()

print("Best model:", best_model_name)

## 19. Residual Analysis

Residuals show the prediction errors.  
I want them to be centered around zero without a strong pattern.

In [ ]:
residuals = y_test - best_predictions

plt.figure(figsize=(10, 6))
sns.histplot(residuals, kde=True, bins=40)
plt.title(f"Residual Distribution — {best_model_name}")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(x=best_predictions, y=residuals, alpha=0.65)
plt.axhline(0, color="red", linestyle="--")
plt.title(f"Residuals vs Predictions — {best_model_name}")
plt.xlabel("Predicted SalePrice")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()

## 20. Error Percentage Distribution

This makes the model error easier to understand in business terms.

In [ ]:
error_percentage = np.abs((y_test - best_predictions) / y_test) * 100

plt.figure(figsize=(10, 6))
sns.histplot(error_percentage, kde=True, bins=40)
plt.title(f"Absolute Percentage Error Distribution — {best_model_name}")
plt.xlabel("Absolute Percentage Error (%)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

print(f"Median absolute percentage error: {np.median(error_percentage):.2f}%")
print(f"Mean absolute percentage error: {np.mean(error_percentage):.2f}%")

# Feature Selection

Now I add a full feature selection section.

This is where the project becomes more advanced.  
Instead of simply training models on every feature, I test which variables actually matter.

## 21. Prepare Encoded Feature Matrix

For feature selection, I transform the data into a numeric matrix first.

In [ ]:
preprocessor_fs = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_features)
    ]
)

X_train_encoded = preprocessor_fs.fit_transform(X_train)
X_test_encoded = preprocessor_fs.transform(X_test)

feature_names = preprocessor_fs.get_feature_names_out()

X_train_encoded = pd.DataFrame(X_train_encoded, columns=feature_names, index=X_train.index)
X_test_encoded = pd.DataFrame(X_test_encoded, columns=feature_names, index=X_test.index)

print("Encoded training data:", X_train_encoded.shape)
print("Encoded testing data:", X_test_encoded.shape)

## 22. Variance Threshold

Variance Threshold removes features that barely change.  
If a feature is almost constant, it usually does not help much.

In [ ]:
variance_selector = VarianceThreshold(threshold=0.01)

X_train_var = variance_selector.fit_transform(X_train_encoded)
X_test_var = variance_selector.transform(X_test_encoded)

selected_var_features = X_train_encoded.columns[variance_selector.get_support()].tolist()

print("Features before variance filtering:", X_train_encoded.shape[1])
print("Features after variance filtering:", len(selected_var_features))

## 23. Correlation Filtering

Highly correlated features can duplicate information.  
Here I remove one feature from pairs with correlation above 0.90.

In [ ]:
corr_matrix = X_train_encoded.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_features = [
    column for column in upper_triangle.columns
    if any(upper_triangle[column] > 0.90)
]

X_train_corr = X_train_encoded.drop(columns=high_corr_features)
X_test_corr = X_test_encoded.drop(columns=high_corr_features)

print("Highly correlated features removed:", len(high_corr_features))
print("Shape after correlation filtering:", X_train_corr.shape)

## 24. SelectKBest

SelectKBest uses a supervised statistical test to select the strongest features.

In [ ]:
k = min(40, X_train_encoded.shape[1])

kbest_selector = SelectKBest(score_func=f_regression, k=k)

X_train_kbest = kbest_selector.fit_transform(X_train_encoded, y_train)
X_test_kbest = kbest_selector.transform(X_test_encoded)

selected_kbest_features = X_train_encoded.columns[kbest_selector.get_support()].tolist()

print("Selected features with SelectKBest:", len(selected_kbest_features))
selected_kbest_features[:20]

In [ ]:
kbest_scores = pd.DataFrame({
    "Feature": X_train_encoded.columns,
    "Score": kbest_selector.scores_
}).replace([np.inf, -np.inf], np.nan).dropna().sort_values("Score", ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=kbest_scores.head(25), x="Score", y="Feature")
plt.title("Top 25 SelectKBest Feature Scores")
plt.xlabel("F-score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

kbest_scores.head(25)

## 25. SelectFromModel with XGBoost

Here I use XGBoost itself to decide which features are important enough to keep.

In [ ]:
sfm_estimator = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

sfm_selector = SelectFromModel(
    estimator=sfm_estimator,
    threshold="median"
)

X_train_sfm = sfm_selector.fit_transform(X_train_encoded, y_train)
X_test_sfm = sfm_selector.transform(X_test_encoded)

selected_sfm_features = X_train_encoded.columns[sfm_selector.get_support()].tolist()

print("Selected features with SelectFromModel:", len(selected_sfm_features))
selected_sfm_features[:20]

## 26. Recursive Feature Elimination

RFE removes weaker features step by step.  
I use Ridge regression here because it is fast and stable.

In [ ]:
rfe_feature_count = min(30, X_train_encoded.shape[1])

rfe_selector = RFE(
    estimator=Ridge(alpha=10),
    n_features_to_select=rfe_feature_count,
    step=10
)

X_train_rfe = rfe_selector.fit_transform(X_train_encoded, y_train)
X_test_rfe = rfe_selector.transform(X_test_encoded)

selected_rfe_features = X_train_encoded.columns[rfe_selector.get_support()].tolist()

print("Selected features with RFE:", len(selected_rfe_features))
selected_rfe_features[:20]

## 27. Feature Selection Model Comparison

Now I train XGBoost on different selected feature sets and compare the results.

In [ ]:
def evaluate_encoded_model(model_name, X_train_data, X_test_data, y_train, y_test):
    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_data, y_train)
    preds = model.predict(X_test_data)

    return {
        "Model": model_name,
        "Features": X_train_data.shape[1],
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds),
        "Predictions": preds,
        "Estimator": model
    }

fs_results = []

fs_results.append(evaluate_encoded_model(
    "XGBoost - all encoded features",
    X_train_encoded,
    X_test_encoded,
    y_train,
    y_test
))

fs_results.append(evaluate_encoded_model(
    "XGBoost - variance threshold",
    pd.DataFrame(X_train_var),
    pd.DataFrame(X_test_var),
    y_train,
    y_test
))

fs_results.append(evaluate_encoded_model(
    "XGBoost - correlation filtered",
    X_train_corr,
    X_test_corr,
    y_train,
    y_test
))

fs_results.append(evaluate_encoded_model(
    "XGBoost - SelectKBest",
    pd.DataFrame(X_train_kbest),
    pd.DataFrame(X_test_kbest),
    y_train,
    y_test
))

fs_results.append(evaluate_encoded_model(
    "XGBoost - SelectFromModel",
    pd.DataFrame(X_train_sfm),
    pd.DataFrame(X_test_sfm),
    y_train,
    y_test
))

fs_results.append(evaluate_encoded_model(
    "XGBoost - RFE",
    pd.DataFrame(X_train_rfe),
    pd.DataFrame(X_test_rfe),
    y_train,
    y_test
))

fs_comparison = pd.DataFrame([
    {k: v for k, v in result.items() if k not in ["Predictions", "Estimator"]}
    for result in fs_results
]).sort_values("RMSE")

fs_comparison

In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(data=fs_comparison, x="RMSE", y="Model")
plt.title("Feature Selection Methods Compared by RMSE")
plt.xlabel("RMSE")
plt.ylabel("")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 6))
sns.barplot(data=fs_comparison.sort_values("R2", ascending=False), x="R2", y="Model")
plt.title("Feature Selection Methods Compared by R²")
plt.xlabel("R²")
plt.ylabel("")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 6))
sns.barplot(data=fs_comparison.sort_values("Features"), x="Features", y="Model")
plt.title("Number of Features Used by Each Method")
plt.xlabel("Number of Features")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 28. XGBoost Feature Importance

Finally, I look inside the XGBoost model to see which features had the strongest impact.

In [ ]:
xgb_pipeline = xgboost_model
xgb_pipeline.fit(X_train, y_train)

processed_feature_names = xgb_pipeline.named_steps["preprocessor"].get_feature_names_out()
xgb_importances = xgb_pipeline.named_steps["model"].feature_importances_

xgb_feature_importance = pd.DataFrame({
    "Feature": processed_feature_names,
    "Importance": xgb_importances
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=xgb_feature_importance.head(25), x="Importance", y="Feature")
plt.title("Top 25 XGBoost Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

xgb_feature_importance.head(25)

## 29. Final Model Summary

At this point, I have:

- explored the housing dataset visually
- engineered meaningful features
- trained several regression models
- added XGBoost as an advanced model
- compared models using MAE, RMSE and R²
- analyzed prediction errors
- used feature selection to understand which variables matter most

This turns the notebook into a complete regression machine learning project.

In [ ]:
print("Best regular model:")
display(model_results.head(1))

print("Best feature selection version:")
display(fs_comparison.head(1))

# Advanced ML Upgrade

Now I am pushing this project beyond a standard regression notebook.

This section adds:

- cross-validation
- XGBoost hyperparameter tuning
- learning curves
- SHAP explainability
- detailed error analysis
- model saving/loading

This makes the project feel much closer to a real machine learning portfolio project.

## 30. Cross-Validation

A single train-test split can sometimes be lucky or unlucky.  
Cross-validation gives a more reliable estimate of model performance by training and validating the model across multiple folds.

In [ ]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    "Ridge Regression": ridge_model,
    "Random Forest": random_forest_model,
    "XGBoost": xgboost_model
}

cv_results = []

for model_name, model in cv_models.items():
    print(f"Running cross-validation for {model_name}...")

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        },
        n_jobs=-1
    )

    cv_results.append({
        "Model": model_name,
        "CV_MAE_Mean": -scores["test_MAE"].mean(),
        "CV_MAE_Std": scores["test_MAE"].std(),
        "CV_RMSE_Mean": -scores["test_RMSE"].mean(),
        "CV_RMSE_Std": scores["test_RMSE"].std(),
        "CV_R2_Mean": scores["test_R2"].mean(),
        "CV_R2_Std": scores["test_R2"].std()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values("CV_RMSE_Mean")
cv_results_df

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=cv_results_df, x="CV_RMSE_Mean", y="Model")
plt.title("Cross-Validation RMSE Comparison")
plt.xlabel("Mean CV RMSE")
plt.ylabel("")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=cv_results_df.sort_values("CV_R2_Mean", ascending=False), x="CV_R2_Mean", y="Model")
plt.title("Cross-Validation R² Comparison")
plt.xlabel("Mean CV R²")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 31. XGBoost Hyperparameter Tuning

This is one of the biggest upgrades in the project.

Instead of using one manually selected XGBoost configuration, I search across multiple parameter combinations and let cross-validation find a stronger setup.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

xgb_tuning_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

param_distributions = {
    "model__n_estimators": [300, 500, 700, 900],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "model__max_depth": [2, 3, 4, 5],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5],
    "model__reg_alpha": [0, 0.01, 0.1],
    "model__reg_lambda": [1, 2, 5]
}

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_tuning_pipeline,
    param_distributions=param_distributions,
    n_iter=25,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

xgb_random_search.fit(X_train, y_train)

print("Best CV RMSE:", -xgb_random_search.best_score_)
print("Best parameters:")
xgb_random_search.best_params_

In [ ]:
tuned_xgb_model = xgb_random_search.best_estimator_

tuned_xgb_predictions = tuned_xgb_model.predict(X_test)

tuned_xgb_result = {
    "Model": "Tuned XGBoost",
    "MAE": mean_absolute_error(y_test, tuned_xgb_predictions),
    "RMSE": np.sqrt(mean_squared_error(y_test, tuned_xgb_predictions)),
    "R2": r2_score(y_test, tuned_xgb_predictions)
}

comparison_with_tuned = pd.concat([
    model_results,
    pd.DataFrame([tuned_xgb_result])
], ignore_index=True).sort_values("RMSE")

comparison_with_tuned

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=comparison_with_tuned, x="RMSE", y="Model")
plt.title("Final Model Comparison Including Tuned XGBoost")
plt.xlabel("RMSE")
plt.ylabel("")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=comparison_with_tuned.sort_values("R2", ascending=False), x="R2", y="Model")
plt.title("Final R² Comparison Including Tuned XGBoost")
plt.xlabel("R²")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 32. Learning Curve

The learning curve helps me understand whether the model would benefit from more data.

If training and validation scores are far apart, the model may be overfitting.  
If both are poor, the model may be underfitting.

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, validation_scores = learning_curve(
    tuned_xgb_model,
    X,
    y,
    cv=3,
    scoring="neg_root_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 5),
    n_jobs=-1
)

train_rmse = -train_scores
validation_rmse = -validation_scores

learning_curve_df = pd.DataFrame({
    "Training Size": train_sizes,
    "Train RMSE Mean": train_rmse.mean(axis=1),
    "Train RMSE Std": train_rmse.std(axis=1),
    "Validation RMSE Mean": validation_rmse.mean(axis=1),
    "Validation RMSE Std": validation_rmse.std(axis=1)
})

learning_curve_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(learning_curve_df["Training Size"], learning_curve_df["Train RMSE Mean"], marker="o", label="Training RMSE")
plt.plot(learning_curve_df["Training Size"], learning_curve_df["Validation RMSE Mean"], marker="o", label="Validation RMSE")
plt.title("Learning Curve — Tuned XGBoost")
plt.xlabel("Training Set Size")
plt.ylabel("RMSE")
plt.legend()
plt.tight_layout()
plt.show()

## 33. Advanced Error Analysis

Now I look at the cases where the model made the biggest mistakes.

This is important because a model can have a good average score and still fail badly on specific types of houses.

In [ ]:
error_analysis = X_test.copy()
error_analysis["ActualSalePrice"] = y_test
error_analysis["PredictedSalePrice"] = tuned_xgb_predictions
error_analysis["Error"] = error_analysis["ActualSalePrice"] - error_analysis["PredictedSalePrice"]
error_analysis["AbsoluteError"] = error_analysis["Error"].abs()
error_analysis["AbsolutePercentageError"] = (
    error_analysis["AbsoluteError"] / error_analysis["ActualSalePrice"] * 100
)

error_analysis_sorted = error_analysis.sort_values("AbsoluteError", ascending=False)

error_analysis_sorted[
    ["ActualSalePrice", "PredictedSalePrice", "Error", "AbsoluteError", "AbsolutePercentageError"]
].head(15)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=error_analysis["ActualSalePrice"],
    y=error_analysis["AbsolutePercentageError"],
    alpha=0.65
)
plt.title("Absolute Percentage Error by Actual Sale Price")
plt.xlabel("Actual SalePrice")
plt.ylabel("Absolute Percentage Error (%)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(error_analysis["AbsolutePercentageError"], kde=True, bins=40)
plt.title("Tuned XGBoost Absolute Percentage Error Distribution")
plt.xlabel("Absolute Percentage Error (%)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 34. SHAP Explainability

SHAP helps explain how the model makes predictions.

Instead of only saying that XGBoost performs well, I can now inspect which features push predictions higher or lower.

In [ ]:
try:
    import shap
except ImportError:
    import sys
    !{sys.executable} -m pip install shap -q
    import shap

# Transform the test set using the fitted preprocessor from the tuned model.
fitted_preprocessor = tuned_xgb_model.named_steps["preprocessor"]
fitted_xgb = tuned_xgb_model.named_steps["model"]

X_test_transformed = fitted_preprocessor.transform(X_test)

try:
    transformed_feature_names = fitted_preprocessor.get_feature_names_out()
except:
    transformed_feature_names = [f"feature_{i}" for i in range(X_test_transformed.shape[1])]

if hasattr(X_test_transformed, "toarray"):
    X_test_transformed_dense = X_test_transformed.toarray()
else:
    X_test_transformed_dense = X_test_transformed

X_test_shap = pd.DataFrame(
    X_test_transformed_dense,
    columns=transformed_feature_names,
    index=X_test.index
)

# Use a sample for faster SHAP plots in Colab.
X_test_shap_sample = X_test_shap.sample(
    n=min(200, len(X_test_shap)),
    random_state=42
)

explainer = shap.TreeExplainer(fitted_xgb)
shap_values = explainer.shap_values(X_test_shap_sample)

print("SHAP sample shape:", X_test_shap_sample.shape)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_shap_sample,
    plot_type="bar",
    max_display=20
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_shap_sample,
    max_display=20
)

## 35. Save the Final Model

A real ML workflow should not stop at training.  
Here I save the tuned model so it can be reused later without retraining.

In [ ]:
import joblib

MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/tuned_xgboost_housing_model.pkl"

joblib.dump(tuned_xgb_model, MODEL_PATH)

print(f"Model saved successfully to: {MODEL_PATH}")

In [ ]:
loaded_model = joblib.load(MODEL_PATH)

sample_predictions = loaded_model.predict(X_test.head(5))

pd.DataFrame({
    "ActualSalePrice": y_test.head(5).values,
    "LoadedModelPrediction": sample_predictions
})

## 36. Advanced Project Conclusion

After adding cross-validation, hyperparameter tuning, SHAP explanations, learning curves, detailed error analysis and model persistence, this project is now much stronger.

This is no longer just a regression notebook.

It now demonstrates:

- serious model evaluation
- advanced boosting
- tuning workflow
- explainable AI
- reusable model artifacts
- deeper error analysis
- portfolio-level machine learning structure

The biggest upgrade is that I am not only training models.  
I am validating them, explaining them, tuning them and saving them for future use.

# Final Conclusion

This project started as a housing price regression task, but I turned it into a full machine learning workflow.

The most important part is not only the final score.  
The real value is understanding the process:

- what drives house prices
- which features are useful
- how different models behave
- how prediction errors look
- whether feature selection improves the model

XGBoost and Random Forest are the strongest choices here because housing prices are affected by non-linear relationships and interactions between features.

This notebook is now much closer to a real portfolio-style ML project than a simple classroom exercise.